# Forecast adjustment check

Validation-only test of a conservative forecast adjustment before applying the selected policy to the 2024 test year.


## What this is—and is not

This is a small, transparent **robustness check**, not stochastic optimisation. It does not model price scenarios, quantiles, risk preferences or probability distributions inside the optimiser.

For a candidate adjustment \(a\), the scheduling price is:

\[\hat{p}^{conservative}_t = \hat{p}_t - a\]

The realised price is never adjusted and is never used for choosing actions. It is used only afterward to settle the historical simulation.

In [1]:
from pathlib import Path

import cvxpy as cp
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

HOURLY_DATA_PATH = Path("data/belgium_day_ahead_hourly_2015_2024.csv")
TEST_FORECAST_PATH = Path("data/forecast_test_2024.csv")

prices = pd.read_csv(HOURLY_DATA_PATH, parse_dates=["timestamp"])
prices["timestamp"] = pd.to_datetime(prices["timestamp"], utc=True, errors="raise")
prices = prices.sort_values("timestamp").reset_index(drop=True)

assert prices["timestamp"].is_monotonic_increasing
assert not prices["timestamp"].duplicated().any()
prices.head()


,timestamp,price_eur_per_mwh
0,2015-01-04 23:00:00+00:00,36.56
1,2015-01-05 00:00:00+00:00,34.94
2,2015-01-05 01:00:00+00:00,32.19
3,2015-01-05 02:00:00+00:00,28.05
4,2015-01-05 03:00:00+00:00,28.04


## Recreate validation forecasts


In [2]:
def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Build leakage-safe calendar and historical-price features."""
    featured = frame.copy()
    hour = featured["timestamp"].dt.hour
    day_of_week = featured["timestamp"].dt.dayofweek

    featured["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    featured["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    featured["day_of_week_sin"] = np.sin(2 * np.pi * day_of_week / 7)
    featured["day_of_week_cos"] = np.cos(2 * np.pi * day_of_week / 7)

    for lag_hours in [24, 48, 72, 168, 336]:
        featured[f"lag_{lag_hours}h"] = featured["price_eur_per_mwh"].shift(lag_hours)

    historical_price = featured["price_eur_per_mwh"].shift(1)
    featured["rolling_mean_24h"] = historical_price.rolling(24).mean()
    featured["rolling_std_24h"] = historical_price.rolling(24).std()
    return featured

FEATURE_COLUMNS = [
    "hour_sin", "hour_cos", "day_of_week_sin", "day_of_week_cos",
    "lag_24h", "lag_48h", "lag_72h", "lag_168h", "lag_336h",
    "rolling_mean_24h", "rolling_std_24h",
]

featured_prices = add_features(prices).dropna(subset=FEATURE_COLUMNS).copy()
TRAIN_END = pd.Timestamp("2023-01-01", tz="UTC")
VALIDATION_END = pd.Timestamp("2024-01-01", tz="UTC")

train = featured_prices.loc[featured_prices["timestamp"] < TRAIN_END].copy()
validation = featured_prices.loc[
    (featured_prices["timestamp"] >= TRAIN_END)
    & (featured_prices["timestamp"] < VALIDATION_END)
].copy()

model = HistGradientBoostingRegressor(
    loss="absolute_error",
    max_iter=300,
    learning_rate=0.06,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42,
)
model.fit(train[FEATURE_COLUMNS], train["price_eur_per_mwh"])

validation["forecast_model_eur_per_mwh"] = model.predict(validation[FEATURE_COLUMNS])
validation["residual_eur_per_mwh"] = (
    validation["price_eur_per_mwh"] - validation["forecast_model_eur_per_mwh"]
)

assert len(validation) == 365 * 24
validation.head()


,timestamp,price_eur_per_mwh,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos,lag_24h,lag_48h,lag_72h,lag_168h,lag_336h,rolling_mean_24h,rolling_std_24h,forecast_model_eur_per_mwh,residual_eur_per_mwh
70033,2023-01-01 00:00:00+00:00,-1.75,0.000000,1.000000,-0.781831,0.62349,0.22,5.09,-0.52,112.11,324.46,5.510417,9.205234,-1.336023,-0.413977
70034,2023-01-01 01:00:00+00:00,-1.46,0.258819,0.965926,-0.781831,0.62349,0.08,3.09,-0.98,106.65,287.47,5.428333,9.263062,-1.667042,0.207042
70035,2023-01-01 02:00:00+00:00,-5.27,0.500000,0.866025,-0.781831,0.62349,0.02,0.65,-2.06,90.84,242.46,5.364167,9.306951,0.574007,-5.844007
70036,2023-01-01 03:00:00+00:00,-4.41,0.707107,0.707107,-0.781831,0.62349,0.01,0.57,-1.69,85.70,214.26,5.143750,9.499666,2.625556,-7.035556
70037,2023-01-01 04:00:00+00:00,-5.40,0.866025,0.500000,-0.781831,0.62349,-0.02,4.88,-0.59,85.65,199.46,4.959583,9.645249,9.342484,-14.742484


## Diagnose forecast error


In [3]:
residual_summary = pd.Series({
    "observations": len(validation),
    "mae_eur_per_mwh": mean_absolute_error(
        validation["price_eur_per_mwh"], validation["forecast_model_eur_per_mwh"]
    ),
    "rmse_eur_per_mwh": mean_squared_error(
        validation["price_eur_per_mwh"], validation["forecast_model_eur_per_mwh"]
    ) ** 0.5,
    "mean_residual_eur_per_mwh": validation["residual_eur_per_mwh"].mean(),
    "residual_p10_eur_per_mwh": validation["residual_eur_per_mwh"].quantile(0.10),
    "residual_p50_eur_per_mwh": validation["residual_eur_per_mwh"].quantile(0.50),
    "residual_p90_eur_per_mwh": validation["residual_eur_per_mwh"].quantile(0.90),
}).round(2)
residual_summary


observations                 8760.00
mae_eur_per_mwh                17.05
rmse_eur_per_mwh               23.68
mean_residual_eur_per_mwh      -1.24
residual_p10_eur_per_mwh      -28.26
residual_p50_eur_per_mwh        0.08
residual_p90_eur_per_mwh       24.47
dtype: float64

In [4]:
hourly_residuals = (
    validation.assign(hour_utc=validation["timestamp"].dt.hour)
    .groupby("hour_utc")["residual_eur_per_mwh"]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)
hourly_residuals


,count,mean,median,std
hour_utc,,,,
0,365,-1.09,2.57,23.90
1,365,-1.61,1.34,23.92
2,365,-3.52,0.32,24.13
3,365,-2.93,0.18,23.38
4,365,-0.00,2.09,23.29
5,365,2.41,4.07,24.08
6,365,1.96,1.45,21.95
7,365,-0.31,-1.75,21.38
8,365,-3.59,-4.53,23.86


## Define daily dispatch under uncertainty


In [5]:
POWER_LIMIT_MW = 1.0
MIN_SOC_MWH = 0.2
MAX_SOC_MWH = 1.8
CHARGE_EFFICIENCY = 0.92
DISCHARGE_EFFICIENCY = 0.92
WEAR_EUR_PER_MWH_DISCHARGED = 5.0
INITIAL_SOC_MWH = 0.8
TOLERANCE = 1e-7


def simulate_daily_dispatch(
    data: pd.DataFrame, forecast_adjustment_eur_per_mwh: float
) -> pd.DataFrame:
    """Schedule one UTC day at a time using adjusted forecasts and settle at actual prices."""
    logs = []

    for _, day_frame in data.groupby(data["timestamp"].dt.floor("D"), sort=True):
        if len(day_frame) != 24:
            raise ValueError(f"Expected 24 rows per UTC day, received {len(day_frame)}")

        adjusted_forecast = (
            day_frame["forecast_model_eur_per_mwh"].to_numpy(dtype=float)
            - forecast_adjustment_eur_per_mwh
        )
        actual_price = day_frame["price_eur_per_mwh"].to_numpy(dtype=float)
        hours = len(day_frame)

        charge_mwh = cp.Variable(hours, nonneg=True)
        discharge_mwh = cp.Variable(hours, nonneg=True)
        soc_mwh = INITIAL_SOC_MWH + cp.cumsum(
            CHARGE_EFFICIENCY * charge_mwh
            - discharge_mwh / DISCHARGE_EFFICIENCY
        )

        constraints = [
            charge_mwh <= POWER_LIMIT_MW,
            discharge_mwh <= POWER_LIMIT_MW,
            soc_mwh >= MIN_SOC_MWH,
            soc_mwh <= MAX_SOC_MWH,
            soc_mwh[-1] == INITIAL_SOC_MWH,
        ]
        objective = cp.Maximize(
            adjusted_forecast @ (discharge_mwh - charge_mwh)
            - WEAR_EUR_PER_MWH_DISCHARGED * cp.sum(discharge_mwh)
        )
        problem = cp.Problem(objective, constraints)
        problem.solve(solver="CLARABEL")

        if problem.status not in {"optimal", "optimal_inaccurate"}:
            raise RuntimeError(f"Solver failed: {problem.status}")

        logs.append(pd.DataFrame({
            "timestamp": day_frame["timestamp"].to_numpy(),
            "charge_mwh": charge_mwh.value,
            "discharge_mwh": discharge_mwh.value,
            "soc_mwh": soc_mwh.value,
            "realised_contribution_eur": (
                actual_price * (discharge_mwh.value - charge_mwh.value)
                - WEAR_EUR_PER_MWH_DISCHARGED * discharge_mwh.value
            ),
            "solver_status": problem.status,
        }))

    return pd.concat(logs, ignore_index=True)


def validate_dispatch(log: pd.DataFrame) -> None:
    """Assert physical feasibility and daily terminal SoC for every simulated action."""
    assert set(log["solver_status"]).issubset({"optimal", "optimal_inaccurate"})
    assert (log["charge_mwh"] >= -TOLERANCE).all()
    assert (log["discharge_mwh"] >= -TOLERANCE).all()
    assert (log["charge_mwh"] <= POWER_LIMIT_MW + TOLERANCE).all()
    assert (log["discharge_mwh"] <= POWER_LIMIT_MW + TOLERANCE).all()
    assert (log["soc_mwh"] >= MIN_SOC_MWH - TOLERANCE).all()
    assert (log["soc_mwh"] <= MAX_SOC_MWH + TOLERANCE).all()

    terminal_soc = log.groupby(log["timestamp"].dt.floor("D"))["soc_mwh"].last()
    assert np.isclose(terminal_soc, INITIAL_SOC_MWH, atol=TOLERANCE).all()
    assert np.minimum(log["charge_mwh"], log["discharge_mwh"]).max() < TOLERANCE


## Select on validation only


In [6]:
CANDIDATE_ADJUSTMENTS_EUR_PER_MWH = [0.0, 2.0, 5.0, 10.0, 15.0, 20.0]

validation_candidates = []
for adjustment in CANDIDATE_ADJUSTMENTS_EUR_PER_MWH:
    candidate_log = simulate_daily_dispatch(validation, adjustment)
    validate_dispatch(candidate_log)
    validation_candidates.append({
        "forecast_adjustment_eur_per_mwh": adjustment,
        "validation_realised_contribution_eur": candidate_log[
            "realised_contribution_eur"
        ].sum(),
        "discharged_mwh": candidate_log["discharge_mwh"].sum(),
    })

validation_policy_results = pd.DataFrame(validation_candidates).sort_values(
    "forecast_adjustment_eur_per_mwh"
).reset_index(drop=True)
validation_policy_results[["validation_realised_contribution_eur", "discharged_mwh"]] = (
    validation_policy_results[["validation_realised_contribution_eur", "discharged_mwh"]].round(2)
)
validation_policy_results


,forecast_adjustment_eur_per_mwh,validation_realised_contribution_eur,discharged_mwh
0,0.0,30438.60,915.62
1,2.0,30365.70,920.83
2,5.0,30300.18,927.60
3,10.0,30267.49,937.59
4,15.0,30205.72,950.23
5,20.0,30243.18,961.00


In [7]:
selected_policy = validation_policy_results.loc[
    validation_policy_results["validation_realised_contribution_eur"].idxmax()
]
selected_adjustment = float(selected_policy["forecast_adjustment_eur_per_mwh"])

print(
    f"Selected adjustment from 2023 validation only: {selected_adjustment:.2f} EUR/MWh"
)
print(
    "Validation realised contribution: "
    f"{selected_policy['validation_realised_contribution_eur']:.2f} EUR"
)


Selected adjustment from 2023 validation only: 0.00 EUR/MWh
Validation realised contribution: 30438.60 EUR


## Evaluate once on 2024


In [8]:
test = pd.read_csv(TEST_FORECAST_PATH, parse_dates=["timestamp"])
test["timestamp"] = pd.to_datetime(test["timestamp"], utc=True, errors="raise")
test = test.sort_values("timestamp").reset_index(drop=True)

selected_test_log = simulate_daily_dispatch(test, selected_adjustment)
validate_dispatch(selected_test_log)

unadjusted_test_log = simulate_daily_dispatch(test, 0.0)
validate_dispatch(unadjusted_test_log)

final_evaluation = pd.DataFrame([
    {
        "policy": "Selected validation policy",
        "forecast_adjustment_eur_per_mwh": selected_adjustment,
        "test_realised_contribution_eur": selected_test_log[
            "realised_contribution_eur"
        ].sum(),
        "test_discharged_mwh": selected_test_log["discharge_mwh"].sum(),
    },
    {
        "policy": "Unadjusted deterministic policy",
        "forecast_adjustment_eur_per_mwh": 0.0,
        "test_realised_contribution_eur": unadjusted_test_log[
            "realised_contribution_eur"
        ].sum(),
        "test_discharged_mwh": unadjusted_test_log["discharge_mwh"].sum(),
    },
])
final_evaluation[["test_realised_contribution_eur", "test_discharged_mwh"]] = (
    final_evaluation[["test_realised_contribution_eur", "test_discharged_mwh"]].round(2)
)
final_evaluation


,policy,forecast_adjustment_eur_per_mwh,test_realised_contribution_eur,test_discharged_mwh
0,Selected validation policy,0.0,33751.14,889.11
1,Unadjusted deterministic policy,0.0,33751.14,889.11
